In [1]:
!pip install -q reductoai

from google.colab import userdata
from reducto import Reducto

client = Reducto(api_key=userdata.get("REDUCTO_API_KEY"))
print("Reducto client ready")

Reducto client ready


In [2]:
from google.colab import files
import json

uploaded = files.upload()   # choose both files in the picker

PDF_PATH = "_10-Q-Q3-2024-As-Filed.pdf"
SCHEMA_PATH = "apple_q3_fy2024_10q_schema.json"

schema = json.load(open(SCHEMA_PATH))["schema"]
print("Fields in schema:", len(schema["properties"]))

Saving apple_q3_fy2024_10q_schema.json to apple_q3_fy2024_10q_schema.json
Saving _10-Q-Q3-2024-As-Filed.pdf to _10-Q-Q3-2024-As-Filed.pdf
Fields in schema: 207


In [3]:
from pathlib import Path

# Upload the PDF to Reducto once; parse and extract both use it
upload = client.upload(file=Path(PDF_PATH))

parse_result = client.parse.run(
    input=upload,
    formatting={"add_page_markers": True, "table_output_format": "html"},
    settings={"timeout": 900, "model": "r-1", "return_citations": True},
)

print("Parse job:", parse_result.job_id)
print("Pages:", parse_result.usage.num_pages)

Parse job: f5836e47-5ac1-4cc3-9858-b5e5f5952054
Pages: 28


In [4]:
SYSTEM_PROMPT = (
    "You are extracting financial data from Apple Inc.'s Form 10-Q for the fiscal quarter "
    "ended June 29, 2024. Follow each field description exactly: use the named table and "
    "period column. Amounts are in USD millions unless the field says otherwise. "
    "Return null when the value is not stated in the document."
)

extract_result = client.extract.run(
    input=upload,
    instructions={"schema": schema, "system_prompt": SYSTEM_PROMPT},
    settings={
        "citations": {"enabled": True, "numerical_confidence": False},
        "alpha": {"citation_model": "v2"},
    },
)

raw = extract_result.model_dump()
fields = raw["result"]

# Big results sometimes come back as a download link instead of the data itself
if isinstance(fields, dict) and "url" in fields and "q3_fy2024_total_net_sales" not in fields:
    import requests
    fields = requests.get(fields["url"]).json()
if isinstance(fields, list):
    fields = fields[0]

json.dump(fields, open("apple_extract_api.json", "w"), indent=2)

print("Extract job:", raw["job_id"])
print("Fields returned:", len(fields))
print("Total net sales Q3 FY24:", fields["q3_fy2024_total_net_sales"]["value"])

Extract job: 43142753-f219-4c3f-99e0-1953afa87f3a
Fields returned: 207
Total net sales Q3 FY24: 85777


In [6]:
!pip install -q pdfplumber

uploaded = files.upload()   # choose both files in the picker

import verify_against_pdf as v

expected = json.load(open("apple_q3_fy2024_expected.json"))["values"]
print("Answer key entries:", len(expected))

Saving verify_against_pdf.py to verify_against_pdf.py
Saving apple_q3_fy2024_expected.json to apple_q3_fy2024_expected.json
Answer key entries: 201


In [7]:
# Pull just the answer out of each field (drop the citation details)
values = {name: (f.get("value") if isinstance(f, dict) else f) for name, f in fields.items()}

rows = v.compare_to_expected(values, expected)
matches = sum(1 for field, got, exp, ok in rows if ok)

print(f"Matches: {matches} of {len(rows)}")
for field, got, exp, ok in rows:
    if not ok:
        print("MISMATCH:", field, "| Reducto:", got, "| answer key:", exp)

Matches: 201 of 201


In [8]:
pdf_rows = v.verify_payload(fields, PDF_PATH)

print(v.summary(pdf_rows))
for r in pdf_rows:
    if r["status"] != "ON PAGE":
        print(r["status"], "|", r["field"], "|", r["value"], "| cited page:", r["page"], "|", r["matched"])

{'ON PAGE': 199, 'NULL': 2}
NULL | ytd_fy2024_term_debt_proceeds | None | cited page: 8 | 
NULL | quantitative_guidance | None | cited page: None | 


In [9]:
# Get the shared helpers from your Amazon builder on GitHub
!wget -q -O build_workbook.py https://raw.githubusercontent.com/DzeGra/reducto-financial-extraction/main/src/build_workbook.py

from openpyxl import Workbook
from build_workbook import Sheet, growth, source_page, FMT_M, FMT_EPS, FMT_PCT, FMT_PP

raw = fields
v = {k: (f.get("value") if isinstance(f, dict) else f) for k, f in fields.items()}

COMPANY = "Apple Inc."
SOURCE = "Form 10-Q, quarter ended June 29, 2024"
PERIODS = ["q3_fy2023", "q3_fy2024", "ytd_fy2023", "ytd_fy2024"]
HEAD = ["Metric", "Q3 FY23", "Q3 FY24", "Q3 YoY", "9M FY23", "9M FY24", "9M YoY", "Page", "Field ID"]
FMT_SH = "#,##0"

def period_row(s, name, key, fmt, bold=False):
    """One metric across the four periods, with YoY growth formulas."""
    a, b, c, d = [v.get(f"{p}_{key}") for p in PERIODS]
    r = s.row
    page = source_page(raw.get(f"q3_fy2024_{key}") or raw.get(f"ytd_fy2024_{key}"))
    s.add([name, a, b, growth(f"C{r}", f"B{r}") if a and b else None,
           c, d, growth(f"F{r}", f"E{r}") if c and d else None, page, f"*_{key}"],
          [None, fmt, fmt, FMT_PCT, fmt, fmt, FMT_PCT], bold=bold)
    return r

def ratio_row(s, name, top, bottom):
    """A calculated margin: row 'top' divided by row 'bottom', change in percentage points."""
    r = s.row
    s.add([name, f"=B{top}/B{bottom}", f"=C{top}/C{bottom}", f"=(C{r}-B{r})*100",
           f"=E{top}/E{bottom}", f"=F{top}/F{bottom}", f"=(F{r}-E{r})*100", None, "calculated"],
          [None, FMT_PCT, FMT_PCT, FMT_PP, FMT_PCT, FMT_PCT, FMT_PP])

def financials_tab(wb):
    s = Sheet(wb, "Financials", f"{COMPANY} | Financial Overview",
              f"USD millions except per-share data · Source: {SOURCE}",
              [46, 13, 13, 12, 13, 13, 12, 8, 34])
    s.section("Income statement")
    s.header(HEAD)
    rows = {}
    for name, key, fmt in [
        ("Total net sales", "total_net_sales", FMT_M),
        ("Products net sales", "products_net_sales", FMT_M),
        ("Services net sales", "services_net_sales", FMT_M),
        ("Gross margin", "gross_margin", FMT_M),
        ("Research and development", "research_and_development", FMT_M),
        ("Selling, general and administrative", "selling_general_administrative", FMT_M),
        ("Operating income", "operating_income", FMT_M),
        ("Provision for income taxes", "provision_for_income_taxes", FMT_M),
        ("Net income", "net_income", FMT_M),
        ("Basic EPS", "basic_eps", FMT_EPS),
        ("Diluted EPS", "diluted_eps", FMT_EPS),
        ("Diluted shares (thousands)", "diluted_shares", FMT_SH),
    ]:
        rows[key] = period_row(s, name, key, fmt, bold=key in ("total_net_sales", "net_income"))
        if key == "gross_margin":
            ratio_row(s, "Gross margin % (calculated)", rows["gross_margin"], rows["total_net_sales"])
        if key == "operating_income":
            ratio_row(s, "Operating margin (calculated)", rows["operating_income"], rows["total_net_sales"])
        if key == "provision_for_income_taxes":
            rates = [v.get(f"{p}_effective_tax_rate_pct") for p in PERIODS]
            pct = [x / 100 if x is not None else None for x in rates]
            r = s.row
            s.add(["Effective tax rate (as reported, MD&A)", pct[0], pct[1], f"=(C{r}-B{r})*100",
                   pct[2], pct[3], f"=(F{r}-E{r})*100",
                   source_page(raw.get("q3_fy2024_effective_tax_rate_pct")), "*_effective_tax_rate_pct"],
                  [None, FMT_PCT, FMT_PCT, FMT_PP, FMT_PCT, FMT_PCT, FMT_PP])

    s.gap()
    s.section("Cash flow (a 10-Q reports cash flows for the nine months only)")
    s.header(HEAD)
    ocf = period_row(s, "Operating cash flow", "operating_cash_flow", FMT_M)
    capex = period_row(s, "Capital expenditures (payments for PP&E)", "purchases_ppe", FMT_M)
    r = s.row
    s.add(["Free cash flow (calculated: operating cash flow − capex)", None, None, None,
           f"=E{ocf}-E{capex}", f"=F{ocf}-F{capex}", growth(f"F{r}", f"E{r}"), None, "calculated"],
          [None, FMT_M, FMT_M, FMT_PCT, FMT_M, FMT_M, FMT_PCT], bold=True)
    period_row(s, "Share-based compensation", "share_based_compensation", FMT_M)
    s.ws.freeze_panes = "B4"
    return rows

wb = Workbook()
wb.remove(wb.active)
financials_tab(wb)
OUT = "Apple_Q3_FY2024_LP_Analysis.xlsx"
wb.save(OUT)
files.download(OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from openpyxl.chart import BarChart, Reference
from build_workbook import show_axes

REGIONS = [("Americas", "americas"), ("Europe", "europe"), ("Greater China", "greater_china"),
           ("Japan", "japan"), ("Rest of Asia Pacific", "rest_of_asia_pacific")]
CATEGORIES = [("iPhone", "iphone"), ("Mac", "mac"), ("iPad", "ipad"),
              ("Wearables, Home and Accessories", "wearables_home_accessories"), ("Services", "services_category")]
FMTS = [None, FMT_M, FMT_M, FMT_PCT, FMT_M, FMT_M, FMT_PCT, None, None]

def total_row(s, first, last):
    r = s.row
    s.add(["Total", f"=SUM(B{first}:B{last})", f"=SUM(C{first}:C{last})", growth(f"C{r}", f"B{r}"),
           f"=SUM(E{first}:E{last})", f"=SUM(F{first}:F{last})", growth(f"F{r}", f"E{r}")], FMTS, bold=True)
    return r

def segments_tab(wb):
    s = Sheet(wb, "Segments", f"{COMPANY} | Segment Performance",
              f"USD millions · Source: {SOURCE} (Note 2 Revenue p.9, Note 11 Segments p.15, MD&A p.19)",
              [34, 13, 13, 12, 13, 13, 12, 8, 34, 13, 12])
    head = ["Segment", "Q3 FY23", "Q3 FY24", "Q3 YoY", "9M FY23", "9M FY24", "9M YoY", "Page", "Field ID"]

    # 1. Net sales by geographic segment
    s.section("Net sales by reportable segment (geographic)")
    s.header(head)
    g_first = s.row
    for name, key in REGIONS:
        period_row(s, name, f"{key}_net_sales", FMT_M)
    g_last = s.row - 1
    g_total = total_row(s, g_first, g_last)

    # 2. Segment operating income and margin
    s.gap()
    s.section("Operating income and margin by segment")
    s.header(["Segment", "Q3 FY23 OI", "Q3 FY24 OI", "Q3 FY23 margin", "Q3 FY24 margin",
              "9M FY23 OI", "9M FY24 OI", "9M FY23 margin", "9M FY24 margin", "9M margin change"])
    m_first = s.row
    for i, (name, key) in enumerate(REGIONS):
        r, sr = s.row, g_first + i
        s.add([name, v[f"q3_fy2023_{key}_operating_income"], v[f"q3_fy2024_{key}_operating_income"],
               f"=B{r}/B{sr}", f"=C{r}/C{sr}", v[f"ytd_fy2023_{key}_operating_income"], v[f"ytd_fy2024_{key}_operating_income"],
               f"=F{r}/E{sr}", f"=G{r}/F{sr}", f"=(I{r}-H{r})*100"],
              [None, FMT_M, FMT_M, FMT_PCT, FMT_PCT, FMT_M, FMT_M, FMT_PCT, FMT_PCT, FMT_PP])
    m_last = s.row - 1
    r = s.row
    s.add(["Total segment operating income", f"=SUM(B{m_first}:B{m_last})", f"=SUM(C{m_first}:C{m_last})",
           f"=B{r}/B{g_total}", f"=C{r}/C{g_total}", f"=SUM(F{m_first}:F{m_last})", f"=SUM(G{m_first}:G{m_last})",
           f"=F{r}/E{g_total}", f"=G{r}/F{g_total}", f"=(I{r}-H{r})*100"],
          [None, FMT_M, FMT_M, FMT_PCT, FMT_PCT, FMT_M, FMT_M, FMT_PCT, FMT_PCT, FMT_PP], bold=True)

    # 3. Net sales by product category
    s.gap()
    s.section("Net sales by category (Note 2, Revenue)")
    s.header(["Category"] + head[1:])
    c_first = s.row
    for name, key in CATEGORIES:
        period_row(s, name, f"{key}_net_sales", FMT_M)
    c_last = s.row - 1
    c_total = total_row(s, c_first, c_last)

    # 4. Gross margin by Products and Services
    s.gap()
    s.section("Gross margin by Products and Services (MD&A)")
    s.header(["Metric"] + head[1:])
    for kind in ("products", "services"):
        gm = period_row(s, f"{kind.capitalize()} gross margin", f"{kind}_gross_margin", FMT_M)
        ns = s.row
        s.add([f"{kind.capitalize()} net sales", *[v.get(f"{p}_{kind}_net_sales") for p in PERIODS[:2]], None,
               *[v.get(f"{p}_{kind}_net_sales") for p in PERIODS[2:]]], FMTS)
        ratio_row(s, f"{kind.capitalize()} gross margin % (calculated)", gm, ns)

    # 5. Revenue mix
    s.gap()
    s.section("Revenue mix (share of total net sales)")
    s.header(["Segment / category", "Q3 FY23", "Q3 FY24", "", "9M FY23", "9M FY24"])
    for first, last, tot in [(g_first, g_last, g_total), (c_first, c_last, c_total)]:
        for sr in range(first, last + 1):
            s.add([f"=A{sr}", f"=B{sr}/B${tot}", f"=C{sr}/C${tot}", None, f"=E{sr}/E${tot}", f"=F{sr}/F${tot}"],
                  [None, FMT_PCT, FMT_PCT, None, FMT_PCT, FMT_PCT])

    # Charts
    ws = s.ws
    c1 = BarChart()
    c1.title, c1.y_axis.title, c1.height, c1.width = "Net sales by segment, nine months (USD M)", "USD millions", 7.5, 15
    c1.add_data(Reference(ws, min_col=5, max_col=6, min_row=g_first - 1, max_row=g_last), titles_from_data=True)
    c1.set_categories(Reference(ws, min_col=1, min_row=g_first, max_row=g_last))
    ws.add_chart(show_axes(c1), "L4")

    c2 = BarChart()
    c2.title, c2.height, c2.width = "Operating margin by segment, nine months", 7.5, 15
    c2.y_axis.number_format = "0%"
    c2.add_data(Reference(ws, min_col=8, max_col=9, min_row=m_first - 1, max_row=m_last), titles_from_data=True)
    c2.set_categories(Reference(ws, min_col=1, min_row=m_first, max_row=m_last))
    ws.add_chart(show_axes(c2), "L20")

    c3 = BarChart()
    c3.title, c3.y_axis.title, c3.height, c3.width = "Net sales by category, nine months (USD M)", "USD millions", 7.5, 15
    c3.add_data(Reference(ws, min_col=5, max_col=6, min_row=c_first - 1, max_row=c_last), titles_from_data=True)
    c3.set_categories(Reference(ws, min_col=1, min_row=c_first, max_row=c_last))
    ws.add_chart(show_axes(c3), "L36")
    ws.freeze_panes = "B4"

wb = Workbook()
wb.remove(wb.active)
financials_tab(wb)
segments_tab(wb)
wb.save(OUT)
files.download(OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [11]:
def cash_flow_tab(wb):
    s = Sheet(wb, "Cash Flow", f"{COMPANY} | Cash Flow, Debt Service and Liquidity",
              f"USD millions · Cash outflows shown as positive amounts · Source: {SOURCE} (pp.6, 8, 11, 20, 23)",
              [58, 14, 14, 12, 8, 34])

    def block(title, rows, prefixes=("ytd_fy2023", "ytd_fy2024"), heads=("9M FY23", "9M FY24")):
        """A two-period table; returns {field key: row number}."""
        s.section(title)
        s.header(["Metric", *heads, "Change", "Page", "Field ID"])
        where = {}
        for name, key in rows:
            a, b = v.get(f"{prefixes[0]}_{key}"), v.get(f"{prefixes[1]}_{key}")
            if a is None and b is None:
                continue  # brief: omit data the source does not report
            r = s.row
            where[key] = r
            s.add([name, a, b, growth(f"C{r}", f"B{r}") if a and b else None,
                   source_page(raw.get(f"{prefixes[1]}_{key}")) or source_page(raw.get(f"{prefixes[0]}_{key}")), f"*_{key}"],
                  [None, FMT_M, FMT_M, FMT_PCT])
        return where

    def calc(name, formula_b, formula_c):
        r = s.row
        s.add([name, formula_b, formula_c, growth(f"C{r}", f"B{r}"), None, "calculated"],
              [None, FMT_M, FMT_M, FMT_PCT], bold=True)
        return r

    # 1. Cash flow highlights
    w = block("Cash flow highlights (nine months)", [
        ("Cash generated by operating activities", "operating_cash_flow"),
        ("Depreciation and amortization", "depreciation_amortization"),
        ("Share-based compensation expense", "share_based_compensation"),
        ("Capital expenditures (payments for PP&E)", "purchases_ppe"),
        ("Purchases of marketable securities", "purchases_marketable_securities"),
        ("Proceeds from maturities of marketable securities", "maturities_marketable_securities"),
        ("Proceeds from sales of marketable securities", "sales_marketable_securities"),
        ("Cash generated by investing activities", "investing_cash_flow"),
        ("Cash paid for income taxes, net", "cash_paid_income_taxes"),
    ])
    o, c = w["operating_cash_flow"], w["purchases_ppe"]
    calc("Free cash flow (calculated: operating cash flow − capex)", f"=B{o}-B{c}", f"=C{o}-C{c}")

    # 2. Debt service and capital return
    s.gap()
    w = block("Debt service and capital return (nine months)", [
        ("Repayments of term debt", "term_debt_repayments"),
        ("Proceeds from issuance of term debt, net", "term_debt_proceeds"),
        ("Commercial paper, net (negative = net repayment)", "commercial_paper_net"),
        ("Payments for dividends and dividend equivalents", "dividends_paid"),
        ("Repurchases of common stock", "share_repurchases"),
        ("Taxes paid on net share settlement of equity awards", "net_share_settlement_taxes"),
        ("Cash used in financing activities (negative = outflow)", "financing_cash_flow"),
    ])
    t, cp = w["term_debt_repayments"], w["commercial_paper_net"]
    debt_row = calc("Total debt repayments (calculated: term debt + net commercial paper)",
                    f"=B{t}-MIN(B{cp},0)", f"=C{t}-MIN(C{cp},0)")
    d, rp = w["dividends_paid"], w["share_repurchases"]
    calc("Total capital returned (calculated: dividends + buybacks)", f"=B{d}+B{rp}", f"=C{d}+C{rp}")
    uses = {"capex": c, "debt": debt_row, "dividends": d, "buybacks": rp}

    # 3. Liquidity
    s.gap()
    w = block("Available cash, marketable securities and debt (balance sheet)", [
        ("Cash and cash equivalents", "cash_cash_equivalents"),
        ("Marketable securities, current", "current_marketable_securities"),
        ("Marketable securities, non-current", "non_current_marketable_securities"),
        ("Commercial paper", "commercial_paper"),
        ("Term debt, current", "current_term_debt"),
        ("Term debt, non-current", "non_current_term_debt"),
        ("Of which restricted (State Aid escrow; included in marketable securities above)", "restricted_marketable_securities"),
    ], prefixes=("sep2023", "jun2024"), heads=("Sep 30, 2023", "Jun 29, 2024"))
    a, b2, c2 = w["cash_cash_equivalents"], w["current_marketable_securities"], w["non_current_marketable_securities"]
    cash_row = calc("Total cash and marketable securities (calculated)", f"=B{a}+B{b2}+B{c2}", f"=C{a}+C{b2}+C{c2}")
    p, q, r_ = w["commercial_paper"], w["current_term_debt"], w["non_current_term_debt"]
    debt_total = calc("Total debt (calculated: commercial paper + term debt)", f"=B{p}+B{q}+B{r_}", f"=C{p}+C{q}+C{r_}")
    calc("Net cash (calculated: cash and securities − debt)", f"=B{cash_row}-B{debt_total}", f"=C{cash_row}-C{debt_total}")

    # 4. Quarter-specific capital return and obligations
    s.gap()
    s.section("Capital return program and obligations (as stated for Q3 FY24 / June 29, 2024)")
    s.header(["Item", "Value", "Unit", "", "Page", "Field ID"])
    for name, key, unit, fmt in [
        ("Shares repurchased in Q3 FY24 (dollar amount, MD&A)", "q3_fy2024_share_repurchases_md_and_a", "USD millions", FMT_M),
        ("Shares repurchased in Q3 FY24 (count)", "q3_fy2024_shares_repurchased_thousands", "thousands of shares", FMT_SH),
        ("Quarterly dividend declared per share", "q3_fy2024_dividend_per_share", "USD", FMT_EPS),
        ("Dividends and dividend equivalents declared, Q3 FY24", "q3_fy2024_dividends_declared", "USD millions", FMT_M),
        ("New repurchase program authorized May 2, 2024", "may2024_new_repurchase_program", "USD millions", FMT_M),
        ("Remaining repurchase authorization, June 29, 2024", "jun2024_remaining_repurchase_authorization", "USD millions", FMT_M),
        ("Manufacturing purchase obligations, total", "jun2024_manufacturing_purchase_obligations", "USD millions", FMT_M),
        ("Manufacturing purchase obligations, due within 12 months", "jun2024_manufacturing_purchase_obligations_12m", "USD millions", FMT_M),
    ]:
        if v.get(key) is None:
            continue
        s.add([name, v[key], unit, None, source_page(raw.get(key)), key], [None, fmt])

    # Chart: uses of cash, nine months
    ws = s.ws
    s.gap()
    s.section("Chart data: main uses of cash (nine months)")
    head = s.header(["Use of cash", "9M FY23", "9M FY24"])
    for name, key in [("Capital expenditures", "capex"), ("Debt repayments", "debt"),
                      ("Dividends", "dividends"), ("Share repurchases", "buybacks")]:
        s.add([name, f"=B{uses[key]}", f"=C{uses[key]}"], [None, FMT_M, FMT_M])
    chart = BarChart()
    chart.title, chart.y_axis.title, chart.height, chart.width = "Main uses of cash, nine months (USD M)", "USD millions", 8, 16
    chart.add_data(Reference(ws, min_col=2, max_col=3, min_row=head, max_row=head + 4), titles_from_data=True)
    chart.set_categories(Reference(ws, min_col=1, min_row=head + 1, max_row=head + 4))
    ws.add_chart(show_axes(chart), "H4")
    ws.freeze_panes = "B4"

wb = Workbook()
wb.remove(wb.active)
financials_tab(wb)
segments_tab(wb)
cash_flow_tab(wb)
wb.save(OUT)
files.download(OUT)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>